In [1]:
import os
import re
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import pandera.pandas as pa
from bs4 import BeautifulSoup
from pandera import Check, Column

from asf_mission_data import storage, utils

In [2]:
os.environ["DATA_ROOT"] = "/home/eglucas/Projects/asf_mission_data/tmp/pipeline-dev"
os.environ["DATA_MODE"] = "LOCAL"

## Bronze

Validators to add
- Expected publication date - schedule?
- Excel file is file downloaded (not ODS)

In [3]:
def latest_collection_page_html_soup(collection_url: str) -> BeautifulSoup:
    """Fetch and parse the HTML content of a data publisher's collection page.

    Args:
        collection_url (str): URL of page containing links to downloadable data files.

    Returns:
        BeautifulSoup: Parsed HTML content of the page.
    """
    return BeautifulSoup(utils.fetch_raw_content(collection_url), "html.parser")

In [4]:
collection_url = "https://www.gov.uk/government/collections/heat-pump-deployment-statistics"

latest_collection_page_html_soup = latest_collection_page_html_soup(collection_url)

In [5]:
def latest_release_page_url(latest_collection_page_html_soup: BeautifulSoup, page_link_text: str) -> str:

    for a in latest_collection_page_html_soup.find_all("a", href=True):
        if page_link_text in a.get_text():
            return urljoin(collection_url, a["href"])

    raise ValueError(f"Could not find statistical release page '{page_link_text}' at {collection_url}")

In [6]:
page_link_text = "Heat pump deployment statistics:"

latest_release_page_url = latest_release_page_url(latest_collection_page_html_soup, page_link_text)

In [7]:
def latest_release_page_html_soup(latest_release_page_url: str) -> BeautifulSoup:
    return BeautifulSoup(utils.fetch_raw_content(latest_release_page_url), "html.parser")

In [8]:
latest_release_page_html_soup = latest_release_page_html_soup(latest_release_page_url)

In [9]:
def latest_file_url(latest_release_page_url: str, latest_release_page_html_soup: BeautifulSoup, file_link_text: str) -> str:

    for a in latest_release_page_html_soup.find_all("a", href=True):
        if file_link_text in a.get_text():
            return urljoin(latest_release_page_url, a["href"])

    raise ValueError(f"Could not find dataset '{file_link_text}' at {latest_release_page_url}")

In [10]:
file_link_text = "Heat pump deployment statistics"

latest_file_url = latest_file_url(latest_release_page_url, latest_release_page_html_soup, file_link_text)

In [11]:
def latest_file_content(latest_file_url: str) -> bytes:
    """Fetch the raw content of the latest data file from given URL.

    Used in the extract stage of the ETL pipeline.

    Args:
        latest_file_url (str): URL of latest data file.

    Returns:
        bytes: Raw content of file.
    """
    return utils.fetch_raw_content(latest_file_url)

In [12]:
latest_file_content = latest_file_content(latest_file_url)

Metadata

In [13]:
def latest_filename(latest_file_url: str) -> str:
    return Path(latest_file_url).name

In [14]:
latest_filename = latest_filename(latest_file_url)

In [15]:
def latest_publication_date(latest_release_page_html_soup: BeautifulSoup) -> str:

    match = re.search(r"Published\s+(\d{1,2}\s+\w+\s+\d{4})", latest_release_page_html_soup.get_text())
    if match:
        return match.group(1)

In [16]:
latest_publication_date = latest_publication_date(latest_release_page_html_soup)

In [17]:
# helper function to go into utils


def normalise_date_string(date_str: str) -> str:
    date = datetime.strptime(date_str.strip(), "%d %B %Y")
    return date.strftime("%Y-%m-%d")

In [18]:
def bronze_metadata(
    publisher: str,
    collection_url: str,
    latest_release_page_url: str,
    latest_file_url: str,
    latest_filename: str,
    latest_publication_date: str,
    bronze_ingest_timestamp: str,
    pipeline_version: str,
) -> dict[str, str]:

    return {
        "publisher": publisher,  # config
        "collection_url": collection_url,  # config
        "page_url": latest_release_page_url,  # node
        "file_url": latest_file_url,  # node
        "filename": latest_filename,  # node
        "publication_date": latest_publication_date,  # node
        "bronze_ingest_timestamp": bronze_ingest_timestamp,  # datetime.now in driver config
        "pipeline_version": pipeline_version,  # version in driver config
        "citation": f"Source: {publisher}, {latest_filename}. Published {latest_publication_date}, {latest_release_page_url}.",
    }

In [19]:
bronze_metadata = bronze_metadata(
    publisher="Department for Energy Security and Net Zero",
    collection_url=collection_url,
    latest_release_page_url=latest_release_page_url,
    latest_file_url=latest_file_url,
    latest_filename=latest_filename,
    latest_publication_date=latest_publication_date,
    bronze_ingest_timestamp=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    pipeline_version=version("asf-mission-data"),
)

Saving out

In [20]:
def bronze_heat_pump_deployment_statistics_file(
    dataset_prefix: str,
    latest_file_content: bytes,
    latest_filename: str,
    latest_publication_date: str,
    bronze_metadata: dict,
) -> None:

    storage.ingest_to_bronze(
        layer_prefix="bronze",
        dataset_prefix=dataset_prefix,
        file=latest_file_content,
        filename=latest_filename,
        date_stamp=f"published={normalise_date_string(latest_publication_date)}",
        metadata=bronze_metadata,
    )

In [21]:
dataset_prefix = "heat_pump_deployment_statistics"

bronze_heat_pump_deployment_statistics_file(dataset_prefix, latest_file_content, latest_filename, latest_publication_date, bronze_metadata)

## Silver

Validators to add
- Schema of output dataframes
- TBD

In [22]:
def bronze_heat_pump_deployment_statistics_file(dataset_prefix: str) -> str:
    return storage.locate_latest_bronze(dataset_prefix, "file")

In [23]:
bronze_heat_pump_deployment_statistics_file = bronze_heat_pump_deployment_statistics_file(dataset_prefix)

In [24]:
def bronze_heat_pump_deployment_statistics_metadata(dataset_prefix: str) -> dict[str, str]:
    metadata_uri = storage.locate_latest_bronze(dataset_prefix, "metadata")
    return storage.read_json(metadata_uri)

In [25]:
bronze_heat_pump_deployment_statistics_metadata = bronze_heat_pump_deployment_statistics_metadata(dataset_prefix)

### Notes

In [26]:
def notes_lookup(bronze_heat_pump_deployment_statistics_file: str, sheet_name: str = "Notes") -> dict:
    notes_df = storage.read_excel_sheet(bronze_heat_pump_deployment_statistics_file, sheet_name)
    # locate row index that has table columns
    header_row = notes_df[notes_df["Notes"].str.contains("Note number", na=False)].index[0]

    # create new df with only data
    df = notes_df.iloc[header_row + 1 :].copy()
    df.columns = notes_df.iloc[header_row]
    df = df.reset_index(drop=True)
    df.columns.name = None

    return dict(zip(df["Note number"], df["Note text"], strict=False))

In [27]:
notes_lookup = notes_lookup(bronze_heat_pump_deployment_statistics_file)

### Table 1.1

In [28]:
def table_1_1_df_raw(bronze_heat_pump_deployment_statistics_file: str, sheet_name: str) -> pd.DataFrame:
    return storage.read_excel_sheet(bronze_heat_pump_deployment_statistics_file, sheet_name)

In [29]:
table_1_1_df_raw = table_1_1_df_raw(bronze_heat_pump_deployment_statistics_file, sheet_name="Table 1.1")

In [30]:
def table_1_1_name(table_1_1_df_raw: pd.DataFrame) -> str:
    return table_1_1_df_raw.columns[0]  # VALIDATION NEEDED

In [31]:
table_1_1_name = table_1_1_name(table_1_1_df_raw)

In [32]:
def table_1_1_data_source(table_1_1_df_raw: pd.DataFrame) -> str:
    return table_1_1_df_raw.loc[3].iloc[0]  # VALIDATION NEEDED

In [33]:
table_1_1_data_source = table_1_1_data_source(table_1_1_df_raw)

In [34]:
def table_1_1_df_cleaned(table_1_1_df_raw: pd.DataFrame, table_1_1_name: str) -> pd.DataFrame:
    # locate row index that has table columns
    header_row = table_1_1_df_raw[table_1_1_df_raw[table_1_1_name].str.contains("Installation quarter", na=False)].index[0]

    # create new df with only data
    df = table_1_1_df_raw.iloc[header_row + 1 :].copy()
    df.columns = table_1_1_df_raw.iloc[header_row]
    df = df.reset_index(drop=True)
    df.columns.name = None

    # clean up column names
    df.columns = df.columns.str.strip().str.replace("\n", "", regex=True)

    return df

In [35]:
table_1_1_df_cleaned = table_1_1_df_cleaned(table_1_1_df_raw, table_1_1_name)

In [36]:
def table_1_1_with_notes_df(table_1_1_df_cleaned: pd.DataFrame, notes_lookup: dict) -> pd.DataFrame:

    df = table_1_1_df_cleaned.copy()

    df["Notes"] = ""

    # parse column names and extract notes
    for col in df.columns:
        match = pd.Series(col).str.extract(r"\[(note \d+)\]").iloc[0, 0]
        if pd.notna(match):
            df["Notes"] = df["Notes"] + " " + notes_lookup.get(match, "")
            df = df.rename(columns={col: col.replace(f"[{match}]", "").strip()})

    # parse "Installation quarter" column and extract notes
    df["Note ref"] = df["Installation quarter"].str.extract(r"\[(note \d+)\]")
    df["Note text"] = df["Note ref"].map(notes_lookup)
    df["Notes"] = df.apply(lambda row: row["Notes"] + " " + row["Note text"] if pd.notna(row["Note text"]) else row["Notes"], axis=1)
    df["Installation quarter"] = df["Installation quarter"].str.replace(r"\[note \d+\]", "", regex=True).str.strip()
    df = df.drop(columns=["Note ref", "Note text"])

    return df

In [37]:
table_1_1_with_notes_df = table_1_1_with_notes_df(table_1_1_df_cleaned, notes_lookup)

In [38]:
def table_1_1_df_datetime_quarters(table_1_1_with_notes_df: pd.DataFrame) -> pd.DataFrame:

    df = table_1_1_with_notes_df.copy()

    # Create installation quarter datetime columns

    q_start = {"Q1": 1, "Q2": 4, "Q3": 7, "Q4": 10}
    q_end = {"Q1": 3, "Q2": 6, "Q3": 9, "Q4": 12}

    df[["Installation quarter start", "Installation quarter end"]] = (
        df["Installation quarter"]
        .str.extract(r"(\d{4}) (Q[1-4])")
        .apply(
            lambda x: pd.Series(
                [
                    pd.Timestamp(year=int(x[0]), month=q_start[x[1]], day=1),
                    pd.Timestamp(year=int(x[0]), month=q_end[x[1]], day=1) + pd.offsets.MonthEnd(0),
                ]
            ),
            axis=1,
        )
    )

    return df

In [39]:
table_1_1_df_datetime_quarters = table_1_1_df_datetime_quarters(table_1_1_with_notes_df)

In [40]:
def table_1_1_df_melted(table_1_1_df_datetime_quarters: pd.DataFrame) -> pd.DataFrame:
    df = table_1_1_df_datetime_quarters.copy()
    id_vars = ["Installation quarter", "Installation quarter start", "Installation quarter end", "Notes"]
    value_vars = ["Air source heat pump installations", "Ground/water source heat pump installations", "Total heat pump installations"]
    return pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name="Type", value_name="value")

In [41]:
table_1_1_df_melted = table_1_1_df_melted(table_1_1_df_datetime_quarters)

In [42]:
TABLE_1_1_SCHEMA = pa.DataFrameSchema(
    {
        "Installation quarter": Column(str),
        "Installation quarter start": Column(pd.Timestamp),
        "Installation quarter end": Column(pd.Timestamp),
        "Notes": Column(str),
        "Type": Column(
            str,
            Check.isin(["Air source heat pump installations", " Ground/water source heat pump installations", "Total heat pump installations"]),
        ),
        "value": Column(float),
        "metadata": Column(object),
    },
    strict=True,
)

In [43]:
# output validation here
def silver_table_1_1_df(
    table_1_1_df_melted: pd.DataFrame,
    bronze_heat_pump_deployment_statistics_metadata: dict[str, str],
    table_1_1_name: str,
    table_1_1_data_source: str,
) -> pd.DataFrame:
    df = table_1_1_df_melted.copy()

    # cast data type
    df["value"] = df["value"].astype(int)

    # append to metadata
    bronze_heat_pump_deployment_statistics_metadata["table_name"] = table_1_1_name
    bronze_heat_pump_deployment_statistics_metadata["data_source"] = table_1_1_data_source
    bronze_heat_pump_deployment_statistics_metadata["citation"] = (
        f"Source: {bronze_heat_pump_deployment_statistics_metadata['publisher']}, {bronze_heat_pump_deployment_statistics_metadata['filename']}. {bronze_heat_pump_deployment_statistics_metadata['table_name']}. Published {bronze_heat_pump_deployment_statistics_metadata['publication_date']}, {bronze_heat_pump_deployment_statistics_metadata['page_url']}."
    )

    df["metadata"] = [bronze_heat_pump_deployment_statistics_metadata] * len(df)

    return df

In [44]:
silver_table_1_1_df = silver_table_1_1_df(
    table_1_1_df_melted, bronze_heat_pump_deployment_statistics_metadata, table_1_1_name, table_1_1_data_source
)

In [45]:
silver_table_1_1_df

,Installation quarter,Installation quarter start,Installation quarter end,Notes,Type,value,metadata
0,2018 Q1: January to March,2018-01-01,2018-03-31,Data is recorded against the relevant quarter...,Air source heat pump installations,1343,{'publisher': 'Department for Energy Security ...
1,2018 Q2: April to June,2018-04-01,2018-06-30,Data is recorded against the relevant quarter...,Air source heat pump installations,1365,{'publisher': 'Department for Energy Security ...
2,2018 Q3: July to September,2018-07-01,2018-09-30,Data is recorded against the relevant quarter...,Air source heat pump installations,1753,{'publisher': 'Department for Energy Security ...
3,2018 Q4: October to December,2018-10-01,2018-12-31,Data is recorded against the relevant quarter...,Air source heat pump installations,2204,{'publisher': 'Department for Energy Security ...
4,2019 Q1: January to March,2019-01-01,2019-03-31,Data is recorded against the relevant quarter...,Air source heat pump installations,2149,{'publisher': 'Department for Energy Security ...
...,...,...,...,...,...,...,...
91,2024 Q4: October to December,2024-10-01,2024-12-31,Data is recorded against the relevant quarter...,Total heat pump installations,13381,{'publisher': 'Department for Energy Security ...
92,2025 Q1: January to March,2025-01-01,2025-03-31,Data is recorded against the relevant quarter...,Total heat pump installations,13457,{'publisher': 'Department for Energy Security ...
93,2025 Q2: April to June,2025-04-01,2025-06-30,Data is recorded against the relevant quarter...,Total heat pump installations,12708,{'publisher': 'Department for Energy Security ...
94,2025 Q3: July to September,2025-07-01,2025-09-30,Data is recorded against the relevant quarter...,Total heat pump installations,13040,{'publisher': 'Department for Energy Security ...


In [46]:
def silver_heat_pump_deployment_statistics_table_1_1_parquet(
    silver_table_1_1_df: pd.DataFrame, dataset_prefix: str, latest_publication_date: str, sheet_name: str
) -> str:
    storage.ingest_to_silver(
        dataset_prefix=dataset_prefix,
        df=silver_table_1_1_df,
        df_name=sheet_name.lower().replace(".", "_").replace(" ", "_"),
        date_stamp=f"published={normalise_date_string(latest_publication_date)}",
    )

In [47]:
silver_heat_pump_deployment_statistics_table_1_1_parquet(silver_table_1_1_df, dataset_prefix, latest_publication_date, sheet_name="Table 1.1")

### Table 1.2

In [48]:
def table_1_2_df_raw(bronze_heat_pump_deployment_statistics_file: str, sheet_name: str) -> pd.DataFrame:
    return storage.read_excel_sheet(bronze_heat_pump_deployment_statistics_file, sheet_name)

In [49]:
table_1_2_df_raw = table_1_2_df_raw(bronze_heat_pump_deployment_statistics_file, sheet_name="Table 1.2")

In [50]:
def table_1_2_name(table_1_2_df_raw: pd.DataFrame) -> str:
    return table_1_2_df_raw.columns[0]  # VALIDATION NEEDED

In [51]:
table_1_2_name = table_1_2_name(table_1_2_df_raw)

In [52]:
table_1_2_name

'Table 1.2 - Number of retrofit heat pump installations by region and quarter, UK, 2018 Q1 to 2025 Q4'

In [53]:
def table_1_2_data_source(table_1_2_df_raw: pd.DataFrame) -> str:
    return table_1_2_df_raw.loc[3].iloc[0]  # VALIDATION NEEDED

In [54]:
table_1_2_data_source = table_1_2_data_source(table_1_2_df_raw)

In [55]:
table_1_2_data_source

'Source: Microgeneration Certification Scheme (MCS) Installation Database'

In [56]:
# TODO do we want the Area Codes and Country or Region row?

In [57]:
def area_codes_lookup(table_1_2_df_raw: pd.DataFrame) -> dict[str, str]:

    # locate row index that has area codes
    area_codes_row = table_1_2_df_raw[table_1_2_df_raw[table_1_2_name].str.contains("Area Codes and Country or Region", na=False)].index[0]
    region_names_row = table_1_2_df_raw[table_1_2_df_raw[table_1_2_name].str.contains("Installation quarter", na=False)].index[0]

    # Extract the rows
    area_codes = table_1_2_df_raw.iloc[area_codes_row, 1:]  # skip first column (header)
    region_names = table_1_2_df_raw.iloc[region_names_row, 1:]  # skip first column

    # Clean region names: remove newlines and extra spaces
    region_names = region_names.str.replace("\n", " ", regex=True).str.strip()

    # Drop NaN codes (last column is NaN)
    mask = area_codes.notna()
    area_codes = area_codes[mask]
    region_names = region_names[mask]

    return dict(zip(region_names, area_codes, strict=False))

In [58]:
area_codes_lookup = area_codes_lookup(table_1_2_df_raw)

In [59]:
def table_1_2_df_cleaned(table_1_2_df_raw: pd.DataFrame, table_1_2_name: str) -> pd.DataFrame:

    # locate row index that has table columns
    header_row = table_1_2_df_raw[table_1_2_df_raw[table_1_2_name].str.contains("Installation quarter", na=False)].index[0]

    # create new df with only data
    df = table_1_2_df_raw.loc[header_row + 1 :].copy()
    df.columns = table_1_2_df_raw.loc[header_row]
    df = df.reset_index(drop=True)
    df.columns.name = None

    # clean up column names
    df.columns = df.columns.str.strip().str.replace("\n", "", regex=True)

    return df

In [60]:
table_1_2_df_cleaned = table_1_2_df_cleaned(table_1_2_df_raw, table_1_2_name)

In [61]:
def table_1_2_with_notes_df(table_1_2_df_cleaned: pd.DataFrame, notes_lookup: dict) -> pd.DataFrame:

    df = table_1_2_df_cleaned.copy()

    df["Notes"] = ""

    # parse column names and extract notes
    for col in df.columns:
        match = pd.Series(col).str.extract(r"\[(note \d+)\]").iloc[0, 0]
        if pd.notna(match):
            df["Notes"] = df["Notes"] + " " + notes_lookup.get(match, "")
            df = df.rename(columns={col: col.replace(f"[{match}]", "").strip()})

    # parse "Installation quarter" column and extract notes
    df["Note ref"] = df["Installation quarter"].str.extract(r"\[(note \d+)\]")
    df["Note text"] = df["Note ref"].map(notes_lookup)
    df["Notes"] = df.apply(lambda row: row["Notes"] + " " + row["Note text"] if pd.notna(row["Note text"]) else row["Notes"], axis=1)
    df["Installation quarter"] = df["Installation quarter"].str.replace(r"\[note \d+\]", "", regex=True).str.strip()
    df = df.drop(columns=["Note ref", "Note text"])

    return df

In [62]:
table_1_2_with_notes_df = table_1_2_with_notes_df(table_1_2_df_cleaned, notes_lookup)

In [63]:
def table_1_2_df_datetime_quarters(table_1_2_with_notes_df: pd.DataFrame) -> pd.DataFrame:

    df = table_1_2_with_notes_df.copy()

    # Create installation quarter datetime columns

    q_start = {"Q1": 1, "Q2": 4, "Q3": 7, "Q4": 10}
    q_end = {"Q1": 3, "Q2": 6, "Q3": 9, "Q4": 12}

    df[["Installation quarter start", "Installation quarter end"]] = (
        df["Installation quarter"]
        .str.extract(r"(\d{4}) (Q[1-4])")
        .apply(
            lambda x: pd.Series(
                [
                    pd.Timestamp(year=int(x[0]), month=q_start[x[1]], day=1),
                    pd.Timestamp(year=int(x[0]), month=q_end[x[1]], day=1) + pd.offsets.MonthEnd(0),
                ]
            ),
            axis=1,
        )
    )

    return df

In [64]:
table_1_2_df_datetime_quarters = table_1_2_df_datetime_quarters(table_1_2_with_notes_df)

In [65]:
table_1_2_df_datetime_quarters.head()

,Installation quarter,United Kingdom,England and Wales,England,North East,North West,Yorkshire and The Humber,East Midlands,West Midlands,East,London,South East,South West,Wales,Scotland,Northern Ireland,Unknown,Notes,Installation quarter start,Installation quarter end
0,2018 Q1: January to March,1514,1209,1112,21,40,122,251,88,245,13,155,177,97,302,0,3,Data is recorded against the relevant quarter...,2018-01-01,2018-03-31
1,2018 Q2: April to June,1498,1209,1095,37,52,120,260,66,251,16,108,185,114,284,0,5,Data is recorded against the relevant quarter...,2018-04-01,2018-06-30
2,2018 Q3: July to September,1899,1388,1284,26,42,143,343,60,333,8,139,190,104,506,0,5,Data is recorded against the relevant quarter...,2018-07-01,2018-09-30
3,2018 Q4: October to December,2416,1867,1734,59,65,175,448,86,337,14,263,287,133,542,0,7,Data is recorded against the relevant quarter...,2018-10-01,2018-12-31
4,2019 Q1: January to March,2291,1860,1732,135,55,176,375,105,376,11,197,302,128,428,0,3,Data is recorded against the relevant quarter...,2019-01-01,2019-03-31


In [66]:
def table_1_2_df_melted(table_1_2_df_datetime_quarters: pd.DataFrame) -> pd.DataFrame:
    df = table_1_2_df_datetime_quarters.copy()
    id_vars = ["Installation quarter", "Installation quarter start", "Installation quarter end", "Notes"]
    value_vars = [
        "United Kingdom",
        "England and Wales",
        "England",
        "North East",
        "North West",
        "Yorkshire and The Humber",
        "East Midlands",
        "West Midlands",
        "East",
        "London",
        "South East",
        "South West",
        "Wales",
        "Scotland",
        "Northern Ireland",
        "Unknown",
    ]
    return pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name="Country or Region", value_name="value")

In [67]:
table_1_2_df_melted = table_1_2_df_melted(table_1_2_df_datetime_quarters)

In [68]:
# TODO
TABLE_1_2_SCHEMA = pa.DataFrameSchema(
    {
        "Installation quarter": Column(str),
        "Installation quarter start": Column(pd.Timestamp),
        "Installation quarter end": Column(pd.Timestamp),
        "Notes": Column(str),
        "Type": Column(
            str,
            Check.isin(["TO DO"]),
        ),
        "value": Column(float),
        "metadata": Column(object),
    },
    strict=True,
)

In [69]:
# output validation here
def silver_table_1_2_df(
    table_1_2_df_melted: pd.DataFrame,
    bronze_heat_pump_deployment_statistics_metadata: dict[str, str],
    table_1_2_name: str,
    table_1_2_data_source: str,
    area_codes_lookup: dict[str, str],
) -> pd.DataFrame:
    df = table_1_2_df_melted.copy()

    # cast data type
    df["value"] = df["value"].astype(int)

    # append to metadata
    bronze_heat_pump_deployment_statistics_metadata["table_name"] = table_1_2_name
    bronze_heat_pump_deployment_statistics_metadata["data_source"] = table_1_2_data_source
    bronze_heat_pump_deployment_statistics_metadata["citation"] = (
        f"Source: {bronze_heat_pump_deployment_statistics_metadata['publisher']}, {bronze_heat_pump_deployment_statistics_metadata['filename']}. {bronze_heat_pump_deployment_statistics_metadata['table_name']}. Published {bronze_heat_pump_deployment_statistics_metadata['publication_date']}, {bronze_heat_pump_deployment_statistics_metadata['page_url']}."
    )

    df["metadata"] = [bronze_heat_pump_deployment_statistics_metadata] * len(df)

    # add area code
    df["Area code"] = df["Country or Region"].map(area_codes_lookup)
    df["Area code"] = df["Area code"].fillna("N/A")

    return df

In [70]:
silver_table_1_2_df = silver_table_1_2_df(
    table_1_2_df_melted, bronze_heat_pump_deployment_statistics_metadata, table_1_2_name, table_1_2_data_source, area_codes_lookup
)

In [71]:
silver_table_1_2_df

,Installation quarter,Installation quarter start,Installation quarter end,Notes,Country or Region,value,metadata,Area code
0,2018 Q1: January to March,2018-01-01,2018-03-31,Data is recorded against the relevant quarter...,United Kingdom,1514,{'publisher': 'Department for Energy Security ...,K02000001
1,2018 Q2: April to June,2018-04-01,2018-06-30,Data is recorded against the relevant quarter...,United Kingdom,1498,{'publisher': 'Department for Energy Security ...,K02000001
2,2018 Q3: July to September,2018-07-01,2018-09-30,Data is recorded against the relevant quarter...,United Kingdom,1899,{'publisher': 'Department for Energy Security ...,K02000001
3,2018 Q4: October to December,2018-10-01,2018-12-31,Data is recorded against the relevant quarter...,United Kingdom,2416,{'publisher': 'Department for Energy Security ...,K02000001
4,2019 Q1: January to March,2019-01-01,2019-03-31,Data is recorded against the relevant quarter...,United Kingdom,2291,{'publisher': 'Department for Energy Security ...,K02000001
...,...,...,...,...,...,...,...,...
507,2024 Q4: October to December,2024-10-01,2024-12-31,Data is recorded against the relevant quarter...,Unknown,1,{'publisher': 'Department for Energy Security ...,N/A
508,2025 Q1: January to March,2025-01-01,2025-03-31,Data is recorded against the relevant quarter...,Unknown,2,{'publisher': 'Department for Energy Security ...,N/A
509,2025 Q2: April to June,2025-04-01,2025-06-30,Data is recorded against the relevant quarter...,Unknown,2,{'publisher': 'Department for Energy Security ...,N/A
510,2025 Q3: July to September,2025-07-01,2025-09-30,Data is recorded against the relevant quarter...,Unknown,3,{'publisher': 'Department for Energy Security ...,N/A


In [72]:
def silver_heat_pump_deployment_statistics_table_1_2_parquet(
    silver_table_1_2_df: pd.DataFrame, dataset_prefix: str, latest_publication_date: str, sheet_name: str
) -> str:
    storage.ingest_to_silver(
        dataset_prefix=dataset_prefix,
        df=silver_table_1_2_df,
        df_name=sheet_name.lower().replace(".", "_").replace(" ", "_"),
        date_stamp=f"published={normalise_date_string(latest_publication_date)}",
    )

In [73]:
silver_heat_pump_deployment_statistics_table_1_2_parquet(silver_table_1_2_df, dataset_prefix, latest_publication_date, sheet_name="Table 1.2")